# Schur mode top-7 subnetworks and greedy information-flow trees

This notebook uses the companion script `schur_mode_greedy_trees_script.py` to:

- load the saved Schur modes from `outputs/schur_modes/schur_modes.npz` when available;
- take the top 7 contributors for each mode;
- extract their small source→receiver `M_ij` submatrix;
- pull the matching rows from `mij_netlist`;
- print a greedy directed flow tree for each mode.

Greedy tree rule: start at the largest-loading contributor, then repeatedly add the strongest available directed edge from any reached contributor to one unreached contributor. If the subnetwork is disconnected, start a new component at the next-largest unreached contributor.

## Notebook role in the analysis sequence

**Role:** Downstream structural explanation of Schur modes.

**Inputs:**
- `matrices/mij_matrix.csv`
- `matrices/mij_netlist.csv`
- `outputs/schur_modes/schur_modes.npz` when available from `03_schur_signal.ipynb`

**Outputs:**
- `outputs/schur_mode_greedy_trees/schur_mode_greedy_tree_summary.csv`
- `outputs/schur_mode_greedy_trees/greedy_trees/`
- `outputs/schur_mode_greedy_trees/submatrices/`
- `outputs/schur_mode_greedy_trees/netlist_edges/`

**What this notebook adds:**
- Reduces each Schur mode to a compact top-contributor subnetwork.
- Builds greedy source-to-receiver information-flow trees.
- Supports the case-study synthesis in `06_schur_mode_case_studies.ipynb`.


In [1]:
from pathlib import Path
import pandas as pd

from schur_mode_greedy_trees_script import (
    build_all_mode_greedy_trees,
    print_all_greedy_trees,
    print_greedy_tree,
    save_greedy_tree_outputs,
)

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 100)

## Configuration

Leave `MAX_MODES = None` to process every Schur mode. Set it to a small number like `3` when you only want a fast preview.

In [2]:
MATRIX_PATH = Path("matrices/mij_matrix.csv")
NETLIST_PATH = Path("matrices/mij_netlist.csv")
OUTPUT_DIR = Path("outputs/schur_mode_greedy_trees")

TOP_N = 7
MAX_MODES = None  # use 3 for a quick preview
NORMALIZATION = "spectral_radius"
TARGET_SPECTRAL_RADIUS = 0.95
INCLUDE_SELF = True

# The script will use this saved archive if present; otherwise it recomputes Schur modes.
SCHUR_ARCHIVE_PATH = Path("outputs/schur_modes/schur_modes.npz")

## Build the top-7 subnetworks and greedy trees

In [3]:
results = build_all_mode_greedy_trees(
    matrix_path=MATRIX_PATH,
    netlist_path=NETLIST_PATH,
    top_n=TOP_N,
    normalization=NORMALIZATION,
    target_spectral_radius=TARGET_SPECTRAL_RADIUS,
    include_self=INCLUDE_SELF,
    max_modes=MAX_MODES,
    schur_archive_path=SCHUR_ARCHIVE_PATH,
    prefer_schur_archive=True,
)

summary = save_greedy_tree_outputs(results, OUTPUT_DIR)
summary.head(10)

,mode,eigenvalue_real,eigenvalue_imag,eigenvalue_magnitude,dominant_cell_type,dominant_region,dominant_ei,dominant_loading_magnitude,n_tree_edges,n_components,roots
0,0,0.950000,0.000000e+00,0.950000,MEC LIII Superficial Multipolar Interneuron,MEC,I,0.670392,5,2,MEC LIII Superficial Multipolar Interneuron; CA1 Trilaminar
1,1,0.370060,-1.110223e-16,0.370060,MEC LIII Superficial Multipolar Interneuron,MEC,I,0.641513,4,3,MEC LIII Superficial Multipolar Interneuron; CA1 Perforant Path Associated QuadD; CA1 R Receiving Apical Targeting
2,2,0.028286,-3.099216e-01,0.311210,DG AIPRIM,DG,I,0.520732,4,3,DG AIPRIM; CA1 Trilaminar; MEC LIII Superficial Multipolar Interneuron
3,3,0.028286,3.099216e-01,0.311210,CA1 Trilaminar,CA1,I,0.584300,6,1,CA1 Trilaminar
4,4,-0.310286,-2.395080e-16,0.310286,DG HICAP,DG,I,0.345367,5,2,DG HICAP; CA1 Perforant Path Associated QuadD
5,5,-0.181766,2.043759e-16,0.181766,CA1 Perforant Path Associated,CA1,I,0.370415,4,3,CA1 Perforant Path Associated; MEC LV VI Pyramidal Polymorphic; LEC LVI Multipolar Pyramidal
6,6,0.159322,1.652386e-17,0.159322,DG Axo Axonic,DG,I,0.444091,1,6,DG Axo Axonic; DG HICAP; MEC LV VI Pyramidal Polymorphic; CA1 R Receiving Apical Targeting; CA2 Basket; LEC LVI Mult...
7,7,-0.042630,1.059215e-01,0.114178,CA1 R Receiving Apical Targeting,CA1,I,0.443261,6,1,CA1 R Receiving Apical Targeting
8,8,-0.060965,8.636200e-02,0.105712,CA1 R Receiving Apical Targeting,CA1,I,0.428715,6,1,CA1 R Receiving Apical Targeting
9,9,-0.042630,-1.059215e-01,0.114178,CA3 Basket,CA3,I,0.351438,4,3,CA3 Basket; CA3 Granule; SUB CA1 Projecting Pyramidal


## Print every mode's greedy information-flow tree

In [4]:
print_all_greedy_trees(results)

Mode 0    eigenvalue 0.95 + 0i |lambda|=0.95
Top contributors:
   1. MEC LIII Superficial Multipolar Interneuron [MEC, I] loading=0.6704
   2. CA1 Trilaminar [CA1, I] loading=0.4928
   3. CA1 Perforant Path Associated QuadD [CA1, I] loading=0.2874
   4. CA1 Perforant Path Associated [CA1, I] loading=0.2274
   5. DG Axo Axonic [DG, I] loading=0.1941
   6. DG AIPRIM [DG, I] loading=0.1377
   7. CA1 R Receiving Apical Targeting [CA1, I] loading=0.1298
Greedy tree flow:
  root: MEC LIII Superficial Multipolar Interneuron
  root: CA1 Trilaminar
  CA1 Trilaminar -> CA1 Perforant Path Associated QuadD  m_ij=-607.6 (-, |m_ij|=607.6)
  CA1 Trilaminar -> CA1 Perforant Path Associated  m_ij=-120.7 (-, |m_ij|=120.7)
    CA1 Perforant Path Associated -> DG Axo Axonic  m_ij=-86.94 (-, |m_ij|=86.94)
    CA1 Perforant Path Associated -> DG AIPRIM  m_ij=-43.27 (-, |m_ij|=43.27)
  CA1 Trilaminar -> CA1 R Receiving Apical Targeting  m_ij=-116.7 (-, |m_ij|=116.7)
Mode 1    eigenvalue 0.3701 - 1.11e-16i |l

## Inspect one mode interactively

Change `MODE_TO_INSPECT` and rerun the cells below.

In [5]:
MODE_TO_INSPECT = 0
mode_result = results[MODE_TO_INSPECT]
print_greedy_tree(mode_result)

Mode 0    eigenvalue 0.95 + 0i |lambda|=0.95
Top contributors:
   1. MEC LIII Superficial Multipolar Interneuron [MEC, I] loading=0.6704
   2. CA1 Trilaminar [CA1, I] loading=0.4928
   3. CA1 Perforant Path Associated QuadD [CA1, I] loading=0.2874
   4. CA1 Perforant Path Associated [CA1, I] loading=0.2274
   5. DG Axo Axonic [DG, I] loading=0.1941
   6. DG AIPRIM [DG, I] loading=0.1377
   7. CA1 R Receiving Apical Targeting [CA1, I] loading=0.1298
Greedy tree flow:
  root: MEC LIII Superficial Multipolar Interneuron
  root: CA1 Trilaminar
  CA1 Trilaminar -> CA1 Perforant Path Associated QuadD  m_ij=-607.6 (-, |m_ij|=607.6)
  CA1 Trilaminar -> CA1 Perforant Path Associated  m_ij=-120.7 (-, |m_ij|=120.7)
    CA1 Perforant Path Associated -> DG Axo Axonic  m_ij=-86.94 (-, |m_ij|=86.94)
    CA1 Perforant Path Associated -> DG AIPRIM  m_ij=-43.27 (-, |m_ij|=43.27)
  CA1 Trilaminar -> CA1 R Receiving Apical Targeting  m_ij=-116.7 (-, |m_ij|=116.7)


In [6]:
mode_result.contributors

,rank,cell_type,region,ei,loading_real,loading_imag,loading_magnitude,energy_fraction
0,1,MEC LIII Superficial Multipolar Interneuron,MEC,I,0.670392,0.000046,0.670392,0.449426
1,2,CA1 Trilaminar,CA1,I,0.492789,0.000034,0.492789,0.242841
2,3,CA1 Perforant Path Associated QuadD,CA1,I,0.287424,0.000020,0.287424,0.082612
3,4,CA1 Perforant Path Associated,CA1,I,0.227403,0.000016,0.227403,0.051712
4,5,DG Axo Axonic,DG,I,0.194089,0.000013,0.194089,0.037671
5,6,DG AIPRIM,DG,I,0.137738,0.000010,0.137738,0.018972
6,7,CA1 R Receiving Apical Targeting,CA1,I,0.129840,0.000009,0.129840,0.016859


In [7]:
mode_result.submatrix

,MEC LIII Superficial Multipolar Interneuron,CA1 Trilaminar,CA1 Perforant Path Associated QuadD,CA1 Perforant Path Associated,DG Axo Axonic,DG AIPRIM,CA1 R Receiving Apical Targeting
MEC LIII Superficial Multipolar Interneuron,-4691.989955,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
CA1 Trilaminar,0.000000,-274.665238,-607.617548,-120.717046,0.000000,0.000000,-116.666232
CA1 Perforant Path Associated QuadD,0.000000,0.000000,-16.143895,-16.516903,0.000000,0.000000,0.000000
CA1 Perforant Path Associated,0.000000,0.000000,-271.380441,-272.084165,-86.942357,-43.268115,0.000000
DG Axo Axonic,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
DG AIPRIM,0.000000,0.000000,0.000000,0.000000,-2091.281900,-2739.629027,0.000000
CA1 R Receiving Apical Targeting,0.000000,0.000000,-180.665861,-150.093963,0.000000,0.000000,-83.345715


In [8]:
mode_result.tree_edges

,component,source,receiver,m_ij,abs_m_ij,source_rank,receiver_rank,source_loading_magnitude,receiver_loading_magnitude,depth
0,2,CA1 Trilaminar,CA1 Perforant Path Associated QuadD,-607.617548,607.617548,2,3,0.492789,0.287424,1
1,2,CA1 Trilaminar,CA1 Perforant Path Associated,-120.717046,120.717046,2,4,0.492789,0.227403,1
2,2,CA1 Trilaminar,CA1 R Receiving Apical Targeting,-116.666232,116.666232,2,7,0.492789,0.129840,1
3,2,CA1 Perforant Path Associated,DG Axo Axonic,-86.942357,86.942357,4,5,0.227403,0.194089,2
4,2,CA1 Perforant Path Associated,DG AIPRIM,-43.268115,43.268115,4,6,0.227403,0.137738,2


In [9]:
mode_result.netlist_edges.head(25)

,pre_neuron,post_neuron,pre_ei,post_ei,w_ij,kappa_j,m_ij
0,DG AIPRIM,DG Axo Axonic,i,i,-7725.744949,0.27069,-2091.281900
1,CA1 Trilaminar,CA1 Perforant Path Associated QuadD,i,i,-1795.454016,0.33842,-607.617548
2,CA1 Perforant Path Associated,CA1 Perforant Path Associated QuadD,i,i,-801.904263,0.33842,-271.380441
3,CA1 R Receiving Apical Targeting,CA1 Perforant Path Associated QuadD,i,i,-533.851017,0.33842,-180.665861
4,CA1 R Receiving Apical Targeting,CA1 Perforant Path Associated,i,i,-258.791619,0.57998,-150.093963
5,CA1 Trilaminar,CA1 Perforant Path Associated,i,i,-208.140015,0.57998,-120.717046
6,CA1 Trilaminar,CA1 R Receiving Apical Targeting,i,i,-449.823535,0.25936,-116.666232
7,CA1 Perforant Path Associated,DG Axo Axonic,i,i,-321.187915,0.27069,-86.942357
8,CA1 Perforant Path Associated,DG AIPRIM,i,i,-173.003261,0.25010,-43.268115
9,CA1 Perforant Path Associated QuadD,CA1 Perforant Path Associated,i,i,-28.478402,0.57998,-16.516903


## Output files

The run above writes CSVs under `outputs/schur_mode_greedy_trees/`:

- one top-contributor table per mode;
- one top-7 source→receiver submatrix per mode;
- one filtered netlist edge table per mode;
- one greedy-tree edge table per mode;
- one summary table across modes.